In [0]:
import os
import shutil
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
import uuid
from datetime import datetime, timezone
pd.set_option('display.max_columns', None)

In [0]:
SCHEMA_BEST_PRICES = pa.schema([
    ("label", pa.string()),
    ("airline", pa.string()),
    ("flight_number", pa.string()),
    ("stops", pa.int64()),
    ("duration_minutes", pa.int64()),
    ("total_price", pa.float64()),
    ("currency", pa.string()),
])

SCHEMA_ROUND_TRIP = pa.schema([
    ("label", pa.string()),
    ("airline", pa.string()),
    ("flight_number", pa.string()),
    ("stops", pa.int64()),
    ("duration_minutes", pa.int64()),
    ("total_price", pa.float64()),
    ("currency", pa.string()),
    ("outbound_date", pa.string()),
    ("return_date", pa.string()),
])

In [0]:
def read_silver(table):
    # Read a Silver table and return as pandas DataFrame
    import pandas as pd
    
    try:
        df = spark.read.table(table)
        pdf = df.toPandas()
        print(f"[gold] {table} loaded: {len(pdf)} rows")
        return pdf
        
    except Exception as e:
        print(f"[gold] Silver table not found — run Airflight-Silver first. Error: {e}")
        return pd.DataFrame()

In [0]:
MONITORING_CATALOG = "monitoring"
MONITORING_SCHEMA = "pipeline_ops"



In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType, LongType, BooleanType
from datetime import datetime, timezone

def write_pipeline_metrics(pipeline_name, started_at, rows_processed,
    rows_quarantined, run_id, pipeline_version='1.0',
    slo_target_minutes=45, status='success', error_message=None
):
    # Import the catalog/schema variables from Cell 3
    from __main__ import MONITORING_CATALOG, MONITORING_SCHEMA
    
    # Define the metrics table name inside the function
    METRICS_TABLE = f"{MONITORING_CATALOG}.{MONITORING_SCHEMA}.etl_metrics"
    
    completed_at = datetime.now(timezone.utc)  # Changed from datetime.utcnow()
    duration_minutes = (completed_at - started_at).total_seconds() / 60
    slo_met = duration_minutes <= slo_target_minutes

    # Define the schema explicitly
    schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("pipeline_name", StringType(), False),
        StructField("pipeline_version", StringType(), False),
        StructField("started_at", TimestampType(), False),
        StructField("completed_at", TimestampType(), False),
        StructField("duration_minutes", DoubleType(), False),
        StructField("rows_processed", LongType(), False),
        StructField("rows_quarantined", LongType(), False),
        StructField("slo_target_minutes", IntegerType(), False),
        StructField("slo_met", BooleanType(), False),
        StructField("status", StringType(), False),
        StructField("error_message", StringType(), True)  # Nullable
    ])

    row = Row(
        run_id=run_id,
        pipeline_name=pipeline_name,
        pipeline_version=pipeline_version,
        started_at=started_at,
        completed_at=completed_at,
        duration_minutes=round(duration_minutes, 2),
        rows_processed=rows_processed,
        rows_quarantined=rows_quarantined,
        slo_target_minutes=slo_target_minutes,
        slo_met=slo_met,
        status=status,
        error_message=error_message
    )

    # Create DataFrame with explicit schema
    (spark.createDataFrame([row], schema=schema)
        .write.format('delta')
        .mode('append')
        .saveAsTable(METRICS_TABLE))
    
    return duration_minutes, slo_met

In [0]:
import pandas as pd
import uuid
from datetime import datetime, timezone

# Capture start time at the beginning
start_time = datetime.now(timezone.utc)
run_id = str(uuid.uuid4())

# Read both Silver tables
direct = read_silver("workspace.silver.airflights_direct")
connecting = read_silver("workspace.silver.airflights_connecting")

# Combine into one DataFrame
combined = pd.concat([direct, connecting], ignore_index=True)
print(f"Combined table: {len(combined)} total rows")

# Initialize result for metrics tracking
result = pd.DataFrame()

# Check data
if not combined.empty:
    print(f"\nTrip type breakdown:")
    print(combined["trip_type"].value_counts())
    display(combined.head())
    
    
    if not result.empty:
        print(f"\nFound {len(result)} round-trip combinations")
        display(result)
        
        # Create temp view for materialized view
        result_spark = spark.createDataFrame(result)
        result_spark.createOrReplaceTempView("temp_roundtrips")
    else:
        print("No round-trip combinations found")
else:
    print("No data in Silver tables")

print(f"✓ Pipeline metrics logged for run: {run_id}")

In [0]:
def round_trip_combos(combined):
    # Find cheapest same-airline round-trip by merging outbound + return on matching airport pair
    import pandas as pd
    
    try:
        # Filter outbound flights
        out = combined[combined["trip_type"] == "outbound"].copy()
        # Filter return flights
        ret = combined[combined["trip_type"] == "return"].copy()

        # Merge outbound and return flights on airline and swapped origin/destination
        merged = out.merge(
            ret,
            left_on=["airline", "origin", "destination"],
            right_on=["airline", "destination", "origin"],
            suffixes=("_out", "_ret")
        )

        # Calculate total round-trip price
        merged["total_price"] = merged["price_out"] + merged["price_ret"]

        # Find index of cheapest round-trip for each airline and route
        idx = merged.groupby(["airline", "origin_out", "destination_out"])["total_price"].idxmin()
        best = merged.loc[idx].copy()

        # Build result DataFrame with required columns for round-trip combos
        result = pd.DataFrame({
            "label": best["origin_out"] + " → " + best["destination_out"] + " | " + best["origin_ret"] + " → " + best["destination_ret"],
            "airline": best["airline"],
            "flight_number": best["flight_number_out"] + " + " + best["flight_number_ret"],
            "stops": best["stops_out"] + best["stops_ret"],
            "duration_minutes": best["duration_minutes_out"] + best["duration_minutes_ret"],
            "total_price": best["total_price"],
            "currency": best["currency_out"],
            "outbound_date": best["departure_time_out"],
            "return_date": best["departure_time_ret"],
        })

        # Return DataFrame of best round-trip combos
        return result
        
    except Exception as e:
        print(f"Error in round_trip_combos: {e}")
        return pd.DataFrame()

In [0]:
Result = round_trip_combos(combined)

# Create the gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# Write Gold table with correct three-part name
result_spark = spark.createDataFrame(Result)
result_spark.write.mode("overwrite").saveAsTable("workspace.gold.airflights_roundtrips")

print(f"✓ Wrote {len(Result)} round-trip combinations to workspace.gold.airflights_roundtrips")

# Write pipeline metrics as the last step
write_pipeline_metrics(
    pipeline_name="AirFlight_Gold_RoundTrip",
    pipeline_version="1.0",
    started_at=start_time,
    rows_processed=len(Result),  # Fixed: was len(view) which doesn't exist
    rows_quarantined=0,
    run_id=run_id
)

print(f"✓ Pipeline metrics logged for run: {run_id}")

In [0]:
%sql
Select* from monitoring.pipeline_ops.etl_metrics